# Generating Cluster Labels for the Balance Semantic Chunk

Cluster labels feed the SupCon objective in `StructuredContrastiveLoss`: same
label = positive, different = negative. This notebook demonstrates two
**complementary** methods on the balance group (raw features:
`balance_amount` and `credit_amount` over 24 months):

1. **Method B — cross-feature, cross-time descriptors.** Instead of
   clustering on a single feature's mean (which gives trivial thresholds the
   encoder can shortcut), summarize **every** feature across **all** time
   steps with `mean + std + slope` and cluster on that richer vector.
2. **Method A — soft positive masks.** Replace the hard same-cluster
   indicator with a `(B, B)` float mask in `[0, 1]` from
   - **GMM responsibilities:** `P(i and j share cluster)` under a Gaussian
     Mixture (parametric, scales to large N), or
   - **Gaussian-kernel kNN graph:** pairwise Gaussian similarity sparsified
     to top-k neighbours (non-parametric, respects local geometry).

Method B feeds Method A: the descriptors are the input to either clustering.

Important: the descriptors and the encoder's input are **not the same
thing**. The encoder still sees the full `(24, F)` series and must extract
the relevant signal via attention; the descriptors only define which pairs
should *count as positives* during training.

In [ ]:
import sys, pathlib
REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / 'ts_embed').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch
import torch.nn.functional as F
np.random.seed(0); torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Synthetic balance group features

Two raw features per customer over 24 months:
- `balance_amount` — outstanding balance
- `credit_amount` — credit limit

Three latent customer profiles (level / trend / volatility differ) so we can
later check that the clustering recovers something meaningful.

In [ ]:
N, T = 2000, 24
rng = np.random.default_rng(0)
true_group = rng.integers(0, 3, N)

# (bal_level, bal_trend, bal_vol, cred_level, cred_trend, cred_vol)
profiles = np.array([
    ( 200,  -5,  60, 5000, 30,  80),    # low-utilization transactor
    (1200,   0, 200, 3000, 10,  60),    # revolver
    (2400,  40, 300, 1800,  0,  40),    # high-risk, rising balance
], dtype=np.float32)

t_axis = np.arange(T, dtype=np.float32)[None, :]
balance_amount = np.zeros((N, T), np.float32)
credit_amount  = np.zeros((N, T), np.float32)
for g in range(3):
    m = true_group == g
    n_g = int(m.sum()); p = profiles[g]
    balance_amount[m] = p[0] + p[1] * t_axis + p[2] * rng.standard_normal((n_g, T))
    credit_amount[m]  = p[3] + p[4] * t_axis + p[5] * rng.standard_normal((n_g, T))

# (N, T, F=2) -- same layout as the encoder's balance feature block.
balance_block = np.stack([balance_amount, credit_amount], axis=-1)
print('balance_block shape :', balance_block.shape)
print('latent group counts :', np.bincount(true_group))

## 2. Method B — cross-feature, cross-time descriptors

Per customer and per feature, compute three summary statistics over the
24-month window:

- **mean** — long-run level.
- **std** — temporal volatility.
- **slope** — least-squares trend (centered time axis, so the formula
  reduces to `sum(x * t_centered) / sum(t_centered^2)`).

Stack into a single descriptor and z-score each dimension across customers so
the cluster algorithm isn't dominated by units. With two raw features this
gives a 6-dim descriptor; in production add derived features like
`utilization = balance / credit`, autocorrelation, paydown bursts.

In [ ]:
def build_descriptors(block):
    """block : (N, T, F)  ->  descriptor : (N, 3*F).
    Per feature: temporal mean, std, and least-squares slope."""
    N, T, F = block.shape
    t_centered = np.arange(T, dtype=np.float32) - (T - 1) / 2.0
    t_var = float((t_centered ** 2).sum())
    mean  = block.mean(axis=1)                                       # (N, F)
    std   = block.std(axis=1)                                        # (N, F)
    slope = (block * t_centered[None, :, None]).sum(axis=1) / t_var  # (N, F)
    return np.concatenate([mean, std, slope], axis=1)                # (N, 3*F)

raw_desc = build_descriptors(balance_block)
desc_mean = raw_desc.mean(0, keepdims=True)
desc_std  = raw_desc.std(0, keepdims=True) + 1e-8
descriptors = (raw_desc - desc_mean) / desc_std

print('descriptor shape  :', descriptors.shape)
print('descriptor columns:')
print('  [0] mean_balance   [1] mean_credit')
print('  [2] std_balance    [3] std_credit')
print('  [4] slope_balance  [5] slope_credit')

## 3. Baseline: hard cluster labels (KMeans)

Hard same-cluster-positive labels via KMeans on the descriptors. Kept here
for comparison with the soft variants below; same-cluster recovery vs. the
latent groups is a quick sanity check that the descriptors carry signal.

In [ ]:
K = 3

def kmeans(X, k, seed=0):
    try:
        from sklearn.cluster import KMeans
        return KMeans(k, n_init=10, random_state=seed).fit_predict(X)
    except ImportError:
        rng2 = np.random.default_rng(seed)
        C = X[rng2.choice(len(X), k, replace=False)]
        for _ in range(50):
            lab = ((X[:, None] - C[None]) ** 2).sum(-1).argmin(1)
            C = np.stack([X[lab == j].mean(0) if (lab == j).any() else C[j]
                          for j in range(k)])
        return lab

hard_labels = kmeans(descriptors, K)
print('hard cluster sizes:', np.bincount(hard_labels))

def cluster_recovery(pred, true):
    from itertools import permutations
    k_true = int(true.max() + 1)
    return max(np.mean([p[t] == pr for t, pr in zip(true, pred)])
               for p in permutations(range(k_true)))
print(f'KMeans recovery of latent groups: {cluster_recovery(hard_labels, true_group):.2f}')

## 4. Method A.1 — soft pos_mask via GMM responsibilities

Fit a Gaussian Mixture on the descriptors. The responsibility matrix
`R` of shape `(N, K)` gives `R[i, k] = P(customer i is in cluster k)`. The
soft positive mask is the per-pair cluster-sharing probability:

```
soft_pos[i, j] = sum_k R[i, k] * R[j, k] = P(i and j share a cluster)
```

Properties: values in `[0, 1]`, symmetric, naturally near 1 on the diagonal
(zeroed out inside the loss). **Scales well**: store the small `(N, K)`
responsibility matrix, then compute `R[batch] @ R[batch].T` per training
batch — no need to materialize an `N × N` mask.

In [ ]:
def gmm_responsibilities(X, k, seed=0):
    try:
        from sklearn.mixture import GaussianMixture
        gmm = GaussianMixture(n_components=k, covariance_type='full',
                              random_state=seed, n_init=3, reg_covar=1e-4)
        gmm.fit(X)
        return gmm.predict_proba(X)
    except ImportError as e:
        raise ImportError(
            'GMM requires scikit-learn (pip install scikit-learn)') from e

R = gmm_responsibilities(descriptors, K).astype(np.float32)
print('responsibilities  shape:', R.shape)
print('responsibility row sums (should all be 1):', R.sum(1)[:5])

# Full (N, N) soft mask -- for demonstration. In production at large N, build
# this per-batch on the fly from R[batch_idx].
soft_mask_gmm = R @ R.T                                             # (N, N)
print('soft_mask_gmm  min={:.3f}  max={:.3f}  mean={:.3f}'.format(
    soft_mask_gmm.min(), soft_mask_gmm.max(), soft_mask_gmm.mean()))

# Diagnostic: how many customers are AMBIGUOUS (no single dominant cluster)?
# These benefit most from a soft mask -- a hard label would arbitrarily
# assign them while the soft mask spreads positive weight across clusters.
ambiguous = (R.max(1) < 0.9).mean()
print(f'customers with max responsibility < 0.9: {ambiguous:.2%}')

## 5. Method A.2 — soft pos_mask via Gaussian-kernel kNN graph

Non-parametric alternative. Compute pairwise Gaussian similarity in
descriptor space, pick a bandwidth `sigma` from the kNN-distance
distribution, then **sparsify** by keeping only each customer's top-k
neighbours so the mask is sparse and computationally cheap.

$$
W_{ij} = \exp\!\Big(-\frac{\|d_i - d_j\|^2}{2\sigma^2}\Big)
$$

Use this when the descriptor distribution isn't well-described by Gaussians
(e.g. heavy tails, non-convex clusters), or when you want a sparser graph.

In [ ]:
def pairwise_sq_dist(X):
    """||x_i - x_j||^2 via the BLAS trick (avoids the (N,N,d) broadcast)."""
    sq = (X * X).sum(1)
    D2 = sq[:, None] + sq[None, :] - 2.0 * (X @ X.T)
    return np.maximum(D2, 0.0)

def gaussian_knn_mask(X, k_neighbors=20, sigma=None):
    """(N, N) sparse soft adjacency in [0, 1]. sigma defaults to the median
    distance to the k_neighbors-th nearest neighbour."""
    D2 = pairwise_sq_dist(X)
    if sigma is None:
        # k-th nearest distance per row -> robust bandwidth.
        kth_d2 = np.partition(D2, k_neighbors, axis=1)[:, k_neighbors]
        sigma = float(np.sqrt(np.median(kth_d2)) + 1e-8)
    W = np.exp(-D2 / (2.0 * sigma * sigma))

    # Keep top-k_neighbors per row, zero the rest, then symmetrize.
    keep = np.zeros_like(W, dtype=bool)
    nn = np.argpartition(-W, k_neighbors, axis=1)[:, :k_neighbors]
    rows = np.arange(len(X))[:, None]
    keep[rows, nn] = True
    keep = keep | keep.T
    return (W * keep).astype(np.float32), sigma

soft_mask_knn, sigma = gaussian_knn_mask(descriptors, k_neighbors=20)
print(f'sigma = {sigma:.3f}  (auto-picked from kNN-distance distribution)')
nnz = (soft_mask_knn > 0).mean()
print(f'soft_mask_knn: {nnz:.4f} non-zero fraction, '
      f'mean non-zero weight = {soft_mask_knn[soft_mask_knn > 0].mean():.3f}')

## 6. (Optional) Visualise the two soft masks

Sort customers by their hard KMeans label so within-cluster blocks line up
along the diagonal. A useful soft mask shows bright diagonal blocks and dim
off-blocks.

In [ ]:
try:
    import matplotlib.pyplot as plt
    order = np.argsort(hard_labels)
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].imshow(soft_mask_gmm[order][:, order], vmin=0, vmax=1, cmap='viridis')
    axes[0].set_title('soft pos_mask: GMM responsibilities')
    axes[1].imshow(soft_mask_knn[order][:, order], vmin=0, vmax=1, cmap='viridis')
    axes[1].set_title('soft pos_mask: Gaussian-kernel kNN graph')
    for ax in axes:
        ax.set_xlabel('customer (sorted by hard cluster)')
        ax.set_ylabel('customer')
    plt.tight_layout(); plt.show()
except ImportError:
    print('matplotlib not installed; skipping visualization')

## 7. How to use the soft mask in training

The package's existing `supcon_loss` expects a **bool** positive mask. Below
is a drop-in **soft** variant: pair weights live in `[0, 1]` instead of being
binary. Per batch, slice the precomputed `(N, K)` responsibility matrix (or
the sparse kNN mask) with the batch's row indices to get a `(B, B)` soft
mask, then tile to the `(2B, 2B)` two-view layout the contrastive loss
expects.

In [ ]:
def soft_supcon_loss(feats, pos_weights, temperature):
    """SupCon with SOFT positive weights in [0, 1] (not bool).

    feats        : (M, d) L2-normalized embeddings.
    pos_weights  : (M, M) float; weight of pair (i, j) being a positive.
                   Diagonal is zeroed internally.
    """
    m = feats.size(0)
    sim = (feats @ feats.t()) / temperature
    sim = sim - sim.max(dim=1, keepdim=True).values.detach()       # stability

    self_mask = torch.eye(m, dtype=torch.bool, device=feats.device)
    pos_weights = pos_weights.masked_fill(self_mask, 0.0)

    exp_sim = torch.exp(sim).masked_fill(self_mask, 0.0)
    log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-12)

    w_sum = pos_weights.sum(dim=1)
    valid = w_sum > 1e-8
    if not valid.any():
        return feats.sum() * 0.0
    per_anchor = -(log_prob * pos_weights).sum(dim=1)[valid] / w_sum[valid]
    return per_anchor.mean()


def stacked_soft_pos_mask(base_soft, b, device):
    """Tile a (B, B) soft mask into the (2B, 2B) two-view layout and force
    same-customer cross-view pairs to weight 1.0 (the two views of a
    customer are always positives regardless of the soft mask)."""
    pm = torch.cat([
        torch.cat([base_soft, base_soft], dim=1),
        torch.cat([base_soft, base_soft], dim=1),
    ], dim=0)
    idx = torch.arange(b, device=device)
    pm[idx, idx + b] = 1.0
    pm[idx + b, idx] = 1.0
    return pm


# ---- minimal 'this is how you'd integrate' example -------------------------
# Inputs you would already have in a real training loop:
#   - chunk projections from the encoder for two augmented views, (B, d_chunk)
#   - the precomputed responsibility matrix R or sparse kNN W
# Pretend the projections here come from a dummy random model.
R_torch = torch.from_numpy(R)                               # (N, K)
B = 64
batch_idx = torch.from_numpy(
    np.random.default_rng(1).choice(N, B, replace=False)).long()

# Build (B, B) soft mask on the fly -- O(B^2 K), no need for the N x N matrix.
R_batch  = R_torch[batch_idx]                                # (B, K)
soft_BB  = R_batch @ R_batch.t()                             # (B, B)

proj_dim = 64
rng_t = torch.Generator().manual_seed(0)
proj_a = F.normalize(torch.randn(B, proj_dim, generator=rng_t), dim=-1)
proj_b = F.normalize(torch.randn(B, proj_dim, generator=rng_t), dim=-1)
feats = torch.cat([proj_a, proj_b], dim=0)                   # (2B, proj_dim)
pos_2B = stacked_soft_pos_mask(soft_BB, B, device=feats.device)

loss = soft_supcon_loss(feats, pos_2B, temperature=0.1)
print(f'demo soft-SupCon loss on a random batch: {float(loss):.3f}')
print('   pos_2B shape:', tuple(pos_2B.shape),
      ' | non-zero weight fraction:', float((pos_2B > 0).float().mean().item()))

## Next steps

- **Plug into the structured loss**: build one soft `(B, B)` mask per
  semantic chunk (balance, payment, ...) and use `soft_supcon_loss` per
  chunk. The package's `StructuredContrastiveLoss` currently casts
  `pos_masks` to bool — either swap in this loss inside a custom training
  loop, or extend the package to accept float masks.
- **Scale to 3M accounts**:
  - **GMM** is the easy path: keep the small `(N, K)` responsibility matrix
    in memory and slice per batch — no `N × N` matrix ever exists.
  - **kNN graph**: use FAISS / hnswlib to build the sparse graph once; per
    batch, look up rows by global index.
- **Choosing K**: BIC for GMM, silhouette / elbow of inertia for KMeans.
  Typically 3–10 per aspect is enough; lower K = denser positives, higher K
  = sharper specialisation.
- **Validating the labels**: fit GMM on a train fold, run `predict_proba` on
  a held-out fold, and check the held-out soft mask still shows the block
  structure. If it doesn't, the descriptors don't generalise and you're at
  risk of memorising labels.
- **Richer descriptors**: extend `build_descriptors` with `utilization =
  balance / clip(credit, 1)`, autocorrelation, paydown bursts, max-utilization
  month, etc. The more cross-feature signal each descriptor dimension
  carries, the harder it is for the encoder to shortcut the clustering.